In [ ]:
# General imports
from netsim.netSimPy import *
from netsim.netSimPy.common.evaluators import EventEvaluator, SimpleEvaluator

# Jorge:
from netsim.netSimPy.common.allocators import sap_ff, Variant
from netsim.netSimPy.common.allocators import (
    route_fragmentation,
    least_fragmentation_band_prioritization,
)

# Patricia:
from netsim.netSimPy.common.allocators import (
    most_available_band_all_routes,
    most_available_route,
)

/Users/jbcedeno/Documents/projcts/multiband-gymnasium/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class NetworkEvaluator(EventEvaluator):
    metrics = None

    def __init__(
        self,
        name,
        bands=["C", "S", "L", "E"],
        filename="net.evaluations",
        header=None,
        info_keywords=[],
        override_existing=True,
    ):
        self.metrics = {
            "steps": 0,
            "blockedEvents": 0,
            "attendedByRouteIndex": {},
            "totalAttendedByBand": {},
            "totalAttended": 0,
            "attendedByModulation": {},
            "totalBitRate": 0,
            "bitRateByBand": {},
        }
        super().__init__(
            name + filename,
            header,
            info_keywords=list(self.metrics.keys()) + info_keywords,
            override_existing=override_existing,
        )
        self.bands = bands

        for band in self.bands:
            self.metrics["totalAttendedByBand"][band] = 0
            self.metrics["bitRateByBand"][band] = 0

    def _on_run_end(self, args):
        block = self.metrics["blockedEvents"]
        steps = self.metrics["steps"]
        self.results_writer.write_row(self.metrics)
        print(f"Blocking probability: {round(block/steps, 6)}")

    def _on_update(self, args):
        event: Event = args["event"]
        self.metrics["steps"] = args["steps"]
        network: Network = args.get("network", None)
        if event.getType() != EventType.Departure:
            self.metrics["blockedEvents"] += 1
        if network is not None:
            if event.getType() == EventType.Departure:
                # the event was allocated, get connection associated:
                for con in network.getConnections(event.id):
                    self.metrics["totalAttended"] += 1
                    self.metrics["totalBitRate"] += con.bitRate.getBitRate()
                    band = con.getBand(con.linksID[0])
                    self.metrics["totalAttendedByBand"][band] += 1
                    self.metrics["bitRateByBand"][band] += con.bitRate.getBitRate()
                    modulation = con.getModulationName(con.linksID[0])
                    if modulation not in self.metrics["attendedByModulation"]:
                        self.metrics["attendedByModulation"][modulation] = 1
                    else:
                        self.metrics["attendedByModulation"][modulation] += 1

                    if con.routeIndex is not None:
                        if con.routeIndex not in self.metrics["attendedByRouteIndex"]:
                            self.metrics["attendedByRouteIndex"][con.routeIndex] = 1
                        else:
                            self.metrics["attendedByRouteIndex"][con.routeIndex] += 1

In [ ]:
M_LAMBDA = 200000
network = Network(
    networkFileName="../../../../networks/nsfnet/network.json",
    pathsFileName="../../../../networks/nsfnet/routes.json",
    bitrateFilename="../../../../networks/nsfnet/bitrates_4_bands.json",
)
generator = EventsGenerator(mLambda=M_LAMBDA)

sim_args = dict(
    eventsGenerator=generator,
    network=network,
    allocator=sap_ff(1),
)

simulator_sap = NetworkSimulator(**sim_args)

sim_args1 = dict(
    eventsGenerator=generator,
    network=network,
    allocator=route_fragmentation(1),
)
simulator_rf = NetworkSimulator(**sim_args1)

sim_args2 = dict(
    eventsGenerator=generator,
    network=network,
    allocator=least_fragmentation_band_prioritization(1),
)
simulator_lfbp = NetworkSimulator(**sim_args2)

In [ ]:
simulators = {
    "sap": simulator_sap,
    "rf": simulator_rf,
    "lfbp": simulator_lfbp,
}
N_EVALUATIONS = 2000000
for name, simulator in simulators.items():
    for traffic in [50000, 100000, 150000, 200000, 250000, 300000, 350000, 400000]:
        callback = NetworkEvaluator(f"{name}_{traffic}")
        simulator.reset()
        simulator.setLambda(traffic)
        simulator.run(N_EVALUATIONS, callback)

Blocking probability: 0.0
Blocking probability: 0.000736
Blocking probability: 0.00999
Blocking probability: 0.026152
Blocking probability: 0.044001
Blocking probability: 0.058722
Blocking probability: 0.070225
Blocking probability: 0.080614
Blocking probability: 0.0
Blocking probability: 4.1e-05
Blocking probability: 0.005595
Blocking probability: 0.021005
Blocking probability: 0.035929
Blocking probability: 0.05094
Blocking probability: 0.064817
Blocking probability: 0.076272
Blocking probability: 0.0
Blocking probability: 0.00054
Blocking probability: 0.006781
Blocking probability: 0.022449
Blocking probability: 0.037777
Blocking probability: 0.051342
Blocking probability: 0.064162
Blocking probability: 0.075949
